In [1]:
import pandas as pd
import numpy as np
import math
from scipy.sparse import csr_matrix, coo_matrix, hstack
from sklearn.preprocessing import StandardScaler
from lightfm import LightFM
from lightfm.evaluation import precision_at_k, recall_at_k, auc_score

/opt/anaconda3/envs/lightfm_env/lib/python3.10/site-packages/lightfm/_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


In [2]:
import pyarrow.parquet as pq

interactions_path = '/Users/alesy/kursach/data/interactions_audio_v2.1.parquet'
pf = pq.ParquetFile(interactions_path)
df = next(pf.iter_batches(batch_size=600_000)).to_pandas()


In [3]:
print(df.shape)
print(df['user_id'].nunique(), 'users,', df['song_id'].nunique(), 'songs')

(600000, 24)
21781 users, 14303 songs


In [4]:
user_cat = df['user_id'].astype('category')
song_cat = df['song_id'].astype('category')
df['u_idx'] = user_cat.cat.codes
df['i_idx'] = song_cat.cat.codes

In [5]:
max(user_cat.cat.codes)  

21780

In [6]:
df_shuf = df.sample(frac=1.0, random_state=42).reset_index(drop=True)
df_shuf['rank'] = df_shuf.groupby('user_id').cumcount()
df_shuf['cnt']  = df_shuf.groupby('user_id')['rank'].transform('max') + 1
is_test = df_shuf['rank'] < (df_shuf['cnt'] * 0.2).astype(int)
df_train = df_shuf[~is_test].reset_index(drop=True)
df_test  = df_shuf[is_test].reset_index(drop=True)
print(f'train={len(df_train):,}  test={len(df_test):,}')
df_train.head()

train=488,737  test=111,263


,user_id,song_id,play_count,deezer_id,name,artists,genre_root,genre_sub,duration,release_date,...,loudness,mode,speechiness,tempo,timeSignature,valence,u_idx,i_idx,rank,cnt
0,a3ce4d625c551a67f6d4ff883374de38f9429b90,SORHSUU12AF72A0E9D,1,651480822,Lore of the Arcane,HammerFall,"rock, metal","rock, metal",87,2002-10-27T00:00:00+00:00,...,-12.27,0,0.04,59.72,4,0.28,13972,9760,0,4
1,3a0ea7eac5802f9472313acbe4cfdd335b227647,SOZVCRW12A67ADA0B7,1,1109953,When You Were Young (Jacques Lu Cont's Thin Wh...,The Killers,"alternative, pop, rock","alternative, pop, rock",220,2006-01-01T00:00:00+00:00,...,-3.31,1,0.11,130.43,4,0.32,4934,14195,0,4
2,e313ab2e4943b6ccd29827ce67a29c4b801b6a7e,SOJEYPO12AAA8C6B0E,1,4288422,Ignorance,Paramore,"alternative, rock","alternative, rock",218,2009-09-04T00:00:00+00:00,...,-2.65,0,0.10,170.94,4,0.51,19257,5178,0,2
3,203fe1d1f68370ec5abe54319c842855fd49bb57,SOFGQNB12A6D4F74F8,1,1983649007,Total Eclipse - 2022 Remaster,Iron Maiden,rock,rock,269,2022-11-18T00:00:00+00:00,...,-5.72,0,0.03,139.78,4,0.51,2714,2925,0,2
4,68bd1f2780037a1f6d59757b07b5e6f671c9abc2,SOYQFQP12A6D4FBDC7,1,662883,This Love,Pantera,"rock, metal","rock, metal",393,1992-02-21T00:00:00+00:00,...,-9.33,1,0.05,104.53,4,0.30,8848,13616,0,4


In [7]:
def to_csr(data):
    return coo_matrix(
        (np.ones(len(data), dtype=np.float32),
         (data['u_idx'].values, data['i_idx'].values))
    ).tocsr()

train_mat = to_csr(df_train)
test_mat  = to_csr(df_test)
print('train nnz:', train_mat.nnz, 'test nnz:', test_mat.nnz)

train nnz: 488737 test nnz: 111263


In [8]:
# --- матрица признаков песен ---
# одна строка на песню, в порядке song_cat.cat.categories (совпадает с i_idx)
songs = (df.drop_duplicates('song_id')
           .set_index('i_idx')
           .sort_index())

NUM_COLS = ['danceability','energy','acousticness','instrumentalness',
            'liveness','speechiness','valence','loudness','tempo']
CAT_COLS = ['genre_root','key','mode','timeSignature']

# числовые: z-score, NaN -> медиана
num = songs[NUM_COLS].astype(float)
num = num.fillna(num.median())
num_scaled = StandardScaler().fit_transform(num).astype(np.float32)

# категориальные: one-hot
cat = pd.get_dummies(
    songs[CAT_COLS].astype('string').fillna('NA'),
    prefix=CAT_COLS, dummy_na=False
).astype(np.float32)

# identity — индивидуальный вектор для каждой песни
from scipy.sparse import identity as sp_identity
identity = sp_identity(songs.shape[0], format='csr', dtype=np.float32)

item_features = hstack([
    identity,
    csr_matrix(num_scaled),
    csr_matrix(cat.values),
]).tocsr()
print('item_features:', item_features.shape)

item_features: (14303, 14910)


In [9]:
# --- обучение LightFM (WARP loss — оптимизирует ранжирование) ---
model = LightFM(
    no_components=32,
    loss='warp',
    learning_rate=0.05,
    item_alpha=1e-6,
    user_alpha=1e-6,
    random_state=42,
)
model.fit(
    train_mat,
    item_features=item_features,
    epochs=15,
    num_threads=4,
    verbose=True,
)

Epoch 0
Epoch 1
Epoch 2
Epoch 3
Epoch 4
Epoch 5
Epoch 6
Epoch 7
Epoch 8
Epoch 9
Epoch 10
Epoch 11
Epoch 12
Epoch 13
Epoch 14


In [10]:
# --- Precision@10, Recall@10, AUC (встроенные в LightFM) ---
K = 10
p = precision_at_k(model, test_mat, train_interactions=train_mat,
                   item_features=item_features, k=K, num_threads=4).mean()
r = recall_at_k(model, test_mat, train_interactions=train_mat,
                item_features=item_features, k=K, num_threads=4).mean()
auc = auc_score(model, test_mat, train_interactions=train_mat,
                item_features=item_features, num_threads=4).mean()
print(f'[LightFM]  Precision@{K}: {p:.4f}')
print(f'[LightFM]  Recall@{K}:    {r:.4f}')
print(f'[LightFM]  AUC:           {auc:.4f}')

[LightFM]  Precision@10: 0.0279
[LightFM]  Recall@10:    0.0572
[LightFM]  AUC:           0.8458


In [11]:
# --- Сравнение метрик близости на LightFM-эмбеддингах ---
import sys, os
sys.path.insert(0, os.path.expanduser('~/kursach'))
from distance_metrics import evaluate_all, to_dataframe
from sklearn.preprocessing import normalize
import pandas as pd

# 1. Эмбеддинги юзеров (без user_features передаются прямо в model.user_embeddings)
user_emb = model.user_embeddings.copy().astype(np.float32)   # (n_users, 32)

# 2. Эмбеддинги песен — взвешенная сумма эмбеддингов фичей
song_emb = (item_features @ model.item_embeddings).astype(np.float32)   # (n_items, 32)

print('user_emb:', user_emb.shape, ' song_emb:', song_emb.shape)

# 3. id в порядке строк
user_ids_order = user_cat.cat.categories.to_numpy()
song_ids_order = song_cat.cat.categories.to_numpy()

# --- Train seen и test truth ---
train_seen   = df_train.groupby('user_id')['song_id'].apply(set).to_dict()
test_by_user = df_test.groupby('user_id')['song_id'].apply(set).to_dict()
print(f'train_seen: {len(train_seen)} users, test_by_user: {len(test_by_user)} users')


# 4. Train seen и test truth уже посчитаны ранее как train_seen и test_by_user

# --- 5а. RAW: сырые эмбеддинги, dot имеет смысл (модель училась под него) ---
results_fm_raw = evaluate_all(
    P=user_emb, M=song_emb,
    user_ids=user_ids_order, item_ids=song_ids_order,
    train_seen=train_seen, test_truth=test_by_user, K=10,
)

# --- 5б. NORMALIZED: L2-нормировка, метрики Минковского становятся сопоставимыми ---
user_emb_n = normalize(user_emb, axis=1)
song_emb_n = normalize(song_emb, axis=1)
results_fm_norm = evaluate_all(
    P=user_emb_n, M=song_emb_n,
    user_ids=user_ids_order, item_ids=song_ids_order,
    train_seen=train_seen, test_truth=test_by_user, K=10,
)

# --- сводная таблица ---
table = pd.concat({
    'raw':  to_dataframe(results_fm_raw),
    'norm': to_dataframe(results_fm_norm),
}, names=['mode', 'metric'])
print(table)
table.to_csv('results_fm.csv')


user_emb: (21781, 32)  song_emb: (14303, 32)
train_seen: 21781 users, test_by_user: 20622 users
[17:17:16] [1/5] metric 'dot': computing scores...
[17:17:17]   scores computed in 0.2s, shape=(21781, 14303)
[17:17:17]   copy scores (21781x14303) -> float64
[17:17:19]   masking train interactions...
[17:17:24]   masking done in 5.1s, computing top-K + metrics...
[17:17:24]     5000/21781 users  rate=8127/s  eta=2s
[17:17:25]     10000/21781 users  rate=8162/s  eta=1s
[17:17:26]     15000/21781 users  rate=8083/s  eta=1s
[17:17:26]     20000/21781 users  rate=8206/s  eta=0s
[17:17:26]   done: 20622 users evaluated in 2.6s
[17:17:26]   'dot' result: P@10=0.0135 R@10=0.0266 NDCG@10=0.0228
[17:17:26] [2/5] metric 'cos': computing scores...
[17:17:27]   scores computed in 0.2s, shape=(21781, 14303)
[17:17:27]   copy scores (21781x14303) -> float64
[17:17:28]   masking train interactions...
[17:17:33]   masking done in 5.7s, computing top-K + metrics...
[17:17:34]     5000/21781 users  rate=79

In [12]:
table

precision@10  recall@10   ndcg@10  users_evaluated
mode metric                                                    
raw  dot         0.013476   0.026574  0.022840          20622.0
     cos         0.012346   0.025518  0.020732          20622.0
     l2          0.005455   0.011175  0.009661          20622.0
     l1          0.004951   0.010005  0.008823          20622.0
     linf        0.003506   0.007112  0.006186          20622.0
norm dot         0.012346   0.025518  0.020732          20622.0
     cos         0.012346   0.025518  0.020732          20622.0
     l2          0.012346   0.025518  0.020732          20622.0
     l1          0.010547   0.021793  0.017577          20622.0
     linf        0.007337   0.014230  0.012377          20622.0